# Adam by Hand → Fair Tuning: Colab GPU Execution Pipeline

This notebook executes the GPU-intensive experiments (Parts 3, 4, 5) on a Google Colab T4 runtime,
persisting results idempotently to Google Drive after each individual run.

In [ ]:
# Cell 1: Mount Google Drive
import os
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/adam-fair-tuning')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['ADAMLAB_RESULTS_DIR'] = str(DRIVE_ROOT / 'results')
    print(f'Google Drive mounted successfully at {DRIVE_ROOT}')
except Exception as e:
    print(f'Running in local/offline environment: {e}')
    os.environ['ADAMLAB_RESULTS_DIR'] = 'results'

In [ ]:
# Cell 2: Checkout pinned commit and install adamlab
!git status
!pip install -e . --no-deps
import adamlab
from adamlab.determinism import get_git_sha
print(f'adamlab version: {adamlab.__version__}, commit: {get_git_sha()}')

In [ ]:
# Cell 3: Environment verification & GPU assertion
import torch
from adamlab.determinism import get_environment_metadata

meta = get_environment_metadata()
print('Runtime Metadata:', meta)
if not torch.cuda.is_available():
    print('WARNING: CUDA not detected! Running on CPU fallback.')
else:
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Cell 4: Determinism smoke test on active device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from adamlab.io import RunSpec
from adamlab.train import run_training

spec = RunSpec(
    experiment='determinism_smoke',
    width=64,
    depth=2,
    lr=0.001,
    schedule='cosine',
    total_steps=10,
    warmup_steps=2,
    seed=42,
    device=device,
    use_synthetic=True,
    eval_steps=[10],
)
res1 = run_training(spec, save_to_disk=False)
res2 = run_training(spec, save_to_disk=False)
assert res1.final_train_loss == res2.final_train_loss, 'Determinism check failed!'
print('Determinism test PASSED: Bitwise identical loss curves.')

In [ ]:
# Cell 5: Benchmark and estimated full-sweep runtime
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
timings = {}
for w in [256, 512, 1024]:
    b_spec = RunSpec(
        experiment='benchmark',
        width=w,
        depth=4,
        lr=0.001,
        schedule='cosine',
        total_steps=20,
        warmup_steps=5,
        seed=0,
        device=device,
        use_synthetic=True,
        eval_steps=[],
    )
    t0 = time.time()
    run_training(b_spec, save_to_disk=False)
    dt = time.time() - t0
    step_sec = dt / 20.0
    timings[w] = step_sec
    print(f'Width {w}: {step_sec*1000:.1f} ms/step ({dt:.2f}s for 20 steps)')

# Estimate full matrix (51 trials/width * 300 steps = 15,300 steps/width)
est_hours = sum(timings[w] * 15300 for w in [256, 512, 1024]) / 3600.0
print(f'Total estimated matrix compute time: {est_hours:.2f} hours')

In [ ]:
# Cell 6: Run matrix execution loop (idempotent, resumable)
from scripts.build_run_matrix import generate_run_matrix
from adamlab.train import run_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
matrix = generate_run_matrix(device=device, use_synthetic=False)
print(f'Processing {len(matrix)} runs in deterministic sequence...')

for idx, run_spec in enumerate(matrix, start=1):
    print(f'[{idx}/{len(matrix)}] Executing {run_spec.experiment} (width={run_spec.width}, lr={run_spec.lr:.2e}, seed={run_spec.seed})...')
    run_training(run_spec, save_to_disk=True)

In [ ]:
# Cell 7: Sync results to local repository
!mkdir -p results/runs
!cp -ru /content/drive/MyDrive/adam-fair-tuning/results/* results/ 2>/dev/null || true
print('Results synchronized.')

In [ ]:
# Cell 8: Aggregations, Figures & Final Checks
device = 'cuda' if torch.cuda.is_available() else 'cpu'
!python -m experiments.p5_lr_sweep --device $device
!python -m experiments.p4_cosine_vs_wsd --device $device
!python -m experiments.p3_update_ratio --device $device
!python -m scripts.make_figures
!python -m scripts.render_readme
!python -m scripts.check_readme
print('All Colab pipeline stages completed successfully!')